In [31]:
import pandas as pd
import numpy as np
import requests
from database_insert import save_shipment
import folium
from folium.plugins import HeatMap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [32]:
# API KEYS

OPENWEATHER_KEY = "2afbf96e46d17e41c8adbe87e17e34d9"
TOMTOM_KEY = "obbl4gsimE74QnjkG7gMN8y1RQY1KsGF"

In [33]:
# LOAD DATASET FOR ML TRAINING


data = pd.read_csv(r"C:\Users\iamsu\OneDrive\Documents\Project\Final_main\AI-BASED-EARLY-WARNING-SYSTEM\data\shipment_dataset_odisha_modified.csv").ffill().bfill()

le_origin = LabelEncoder()
le_dest = LabelEncoder()
le_weather = LabelEncoder()
le_traffic = LabelEncoder()
le_carrier = LabelEncoder()

data["origin"] = le_origin.fit_transform(data["origin"].astype(str))
data["destination"] = le_dest.fit_transform(data["destination"].astype(str))
data["weather"] = le_weather.fit_transform(data["weather"].astype(str))
data["traffic"] = le_traffic.fit_transform(data["traffic"].astype(str))
data["Carrier_History"] = le_carrier.fit_transform(data["Carrier_History"].astype(str))

X = data.drop(["shipment_id","delay"],axis=1).astype(np.float32)
y = data["delay"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = XGBClassifier(n_estimators=200,learning_rate=0.6,max_depth=12,subsample=0.9,colsample_bytree=0.9,random_state=42,eval_metric="logloss")
model.fit(X_train,y_train)

print("Model Accuracy:",accuracy_score(y_test,model.predict(X_test)))

Model Accuracy: 0.88


In [34]:
# CITY COORDINATES


city_coords = {

"Delhi": (28.6139,77.2090),
"Mumbai": (19.0760,72.8777),
"Bangalore": (12.9716,77.5946),
"Chennai": (13.0827,80.2707),
"Hyderabad": (17.3850,78.4867),
"Pune": (18.5204,73.8567),
"Ahmedabad": (23.0225,72.5714),
"Kolkata": (22.5726,88.3639),
"Lucknow": (26.8467,80.9462),
"Jaipur": (26.9124,75.7873),
"Puri": (19.8135,85.8312),
"Bhubaneswar": (20.2961,85.8245),
"Cuttack": (20.4625,85.8828),
"Baripada": (21.9333,86.7500),
"Nagpur": (21.1458,79.0882),
"Angul": (20.8399,85.1018),
"Rourkela": (22.2604,84.8536),
"Keonjhar": (21.6231,85.5994),
"Dhenkanal": (20.6602,85.5994),
"Balasore": (21.4938,86.9311),
"Sambalpur": (21.4667,83.9667),
"Berhampur": (19.3144,84.7911),
"Patna": (25.5941,85.1376),
"Varanasi": (25.3176,82.9739),
"Vishakhapatnam": (17.6868,83.2185),
"Jharsuguda": (21.9100,84.0000),
"Raipur": (21.2514,81.6296),
"Baripada": (21.9333,86.7500),
"Ranch": (23.3441,85.3096),

}

In [35]:
# WEATHER FUNCTION

def weather_level(city):

    url=f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_KEY}"

    try:
        r=requests.get(url,timeout=3).json()
        w=r["weather"][0]["main"]
    except:
        return 0

    if w in ["Rain","Thunderstorm"]:
        return 2
    elif w=="Clouds":
        return 1
    else:
        return 0

In [36]:
# TRAFFIC FUNCTION

def traffic_penalty(lat,lon):

    url=f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?point={lat},{lon}&key={TOMTOM_KEY}"

    try:

        r=requests.get(url,timeout=3).json()
        flow=r["flowSegmentData"]

        ratio=flow["currentSpeed"]/flow["freeFlowSpeed"]

        if ratio<0.5:
            return 2
        elif ratio<0.8:
            return 1

    except:
        pass

    return 0

In [37]:
# OSRM ROAD ROUTES

def get_routes(origin,dest):

    mid_lat=(origin[0]+dest[0])/2
    mid_lon=(origin[1]+dest[1])/2

    urls=[

    f"http://router.project-osrm.org/route/v1/driving/{origin[1]},{origin[0]};{dest[1]},{dest[0]}?overview=full&geometries=geojson",

    f"http://router.project-osrm.org/route/v1/driving/{origin[1]},{origin[0]};{mid_lon+0.5},{mid_lat+0.5};{dest[1]},{dest[0]}?overview=full&geometries=geojson"

    ]

    routes=[]

    for u in urls:

        try:

            r=requests.get(u).json()["routes"][0]

            coords=[(c[1],c[0]) for c in r["geometry"]["coordinates"]]

            distance=r["distance"]/1000
            eta=r["duration"]/3600

            routes.append((coords,distance,eta))

        except:
            pass

    return routes

In [38]:
# AI ANOMALY DETECTION

def detect_anomaly(distance, eta, risk):

    issues=[]

    if distance>1800:
        issues.append("Unusually long route")

    if eta>30:
        issues.append("Extremely high travel time")

    if risk>0.7:
        issues.append("High delay probability")

    return issues

In [39]:
# AI DISRUPTION SIMULATION

def simulate_disruption(eta,risk,weather,traffic):

    print("\nAI Disruption Simulation")

    new_eta=eta
    new_risk=risk

    if weather>0:
        new_eta*=1.2
        new_risk+=0.15
        print("Weather disruption simulated")

    if traffic>0:
        new_eta*=1.15
        new_risk+=0.1
        print("Traffic congestion simulated")

    return new_eta,new_risk

In [40]:
# USER INPUT

n=int(input("Enter number of shipments: "))

shipments=[]

for i in range(n):

    print("\nShipment",i+1)

    origin=input("Origin city: ")
    dest=input("Destination city: ")

    shipments.append((origin,dest))


# MAP


map_dashboard=folium.Map(location=[22.5,78.9],zoom_start=5)

risk_points=[]



# PROCESS SHIPMENTS


for origin_city,dest_city in shipments:

    origin=city_coords.get(origin_city)
    dest=city_coords.get(dest_city)

    if origin is None or dest is None:
        print("City not supported")
        continue

    routes=get_routes(origin,dest)

    scores=[]
    route_data=[]

    for coords,dist,eta in routes:

        lat,lon=coords[len(coords)//2]

        traffic=traffic_penalty(lat,lon)
        weather=weather_level(origin_city)

        features=np.array([[

        le_origin.transform([origin_city])[0],
        le_dest.transform([dest_city])[0],
        dist,
        weather,
        traffic,
        2,
        eta,
        1

        ]],dtype=np.float32)

        risk=model.predict_proba(features)[0][1]

        score=dist+eta*10+traffic*20+risk*100

        route_data.append((coords,dist,eta,risk))
        scores.append(score)

        risk_points.append([float(lat),float(lon),float(risk)])

    best=scores.index(min(scores))


    anomaly=detect_anomaly(
    route_data[best][1],
    route_data[best][2],
    route_data[best][3]
    )

    if anomaly:

        print("\nAI Anomaly Detected")

        for a in anomaly:
            print("-",a)


    new_eta,new_risk=simulate_disruption(
    route_data[best][2],
    route_data[best][3],
    weather,
    traffic
    )


    if new_risk > 0.5 and len(routes) > 1:

        print("\n⚠ High disruption risk detected")

        choice=input("Do you want to change the route? (yes/no): ").lower()


        
        # USER CHOOSES REROUTE
   

        if choice=="yes":

            best=1

            reroute_map=folium.Map(location=[22.5,78.9],zoom_start=5)

            coords,dist,eta,risk=route_data[best]

            folium.PolyLine(coords,color="red",weight=7).add_to(reroute_map)

            folium.Marker(origin,popup="Origin",
                          icon=folium.Icon(color="blue")).add_to(reroute_map)

            folium.Marker(dest,popup="Destination",
                          icon=folium.Icon(color="red")).add_to(reroute_map)

            filename=f"reroute_{origin_city}_{dest_city}.html"

            reroute_map.save(filename)

            print("Reroute map saved as",filename)


        
        # USER CHOOSES NO REROUTE
        

        elif choice=="no":

            best_route_map=folium.Map(location=[22.5,78.9],zoom_start=5)

            coords,dist,eta,risk=route_data[best]

            folium.PolyLine(coords,color="green",weight=7).add_to(best_route_map)

            folium.Marker(origin,popup="Origin",
                          icon=folium.Icon(color="blue")).add_to(best_route_map)

            folium.Marker(dest,popup="Destination",
                          icon=folium.Icon(color="red")).add_to(best_route_map)

            filename=f"best_route_{origin_city}_{dest_city}.html"

            best_route_map.save(filename)

            print("Best route map saved as",filename)


    for i,(coords,dist,eta,risk) in enumerate(route_data):

        color="green" if i==best else "blue"

        folium.PolyLine(coords,color=color,weight=6).add_to(map_dashboard)

    folium.Marker(origin,popup="Origin",
                  icon=folium.Icon(color="blue")).add_to(map_dashboard)

    folium.Marker(dest,popup="Destination",
                  icon=folium.Icon(color="red")).add_to(map_dashboard)


    print("\nShipment Result")
    print("Origin:",origin_city)
    print("Destination:",dest_city)
    print("Distance:",round(route_data[best][1],2),"km")
    print("ETA:",round(route_data[best][2],2),"hours")
    print("Delay Risk:",round(route_data[best][3],2))


    save_shipment(
    origin_city,
    dest_city,
    route_data[best][0],
    route_data[best][1],
    route_data[best][2],
    route_data[best][3]
    )

    print("Shipment saved successfully")


HeatMap(risk_points,radius=25).add_to(map_dashboard)


map_dashboard.save("smart_logistics_map.html")

print("\nMap saved as smart_logistics_map.html")


Shipment 1

Shipment 2

AI Disruption Simulation

Shipment Result
Origin: Pune
Destination: Puri
Distance: 1559.48 km
ETA: 18.93 hours
Delay Risk: 0.0
Shipment saved successfully
Shipment saved successfully

AI Anomaly Detected
- High delay probability

AI Disruption Simulation

⚠ High disruption risk detected
Reroute map saved as reroute_Chennai_Kolkata.html

Shipment Result
Origin: Chennai
Destination: Kolkata
Distance: 1687.83 km
ETA: 20.9 hours
Delay Risk: 0.88
Shipment saved successfully
Shipment saved successfully

Map saved as smart_logistics_map.html
